# PEEC Impedance Extraction with ngsbem

This notebook demonstrates the **Partial Element Equivalent Circuit (PEEC)** method
for computing the port impedance $Z(f)$ of a conductor, using **ngsbem** (NGSolve BEM)
for the Galerkin boundary element assembly.

## Physical problem

We consider a thin, flat conductor (e.g., a PCB trace or a bus bar) carrying
alternating current. The goal is to compute the frequency-dependent impedance
$Z(f) = R(f) + j\omega L(f)$, which captures:

- **DC resistance** from the conductor's sheet resistance $R_\square = 1/(\sigma t)$
- **Inductive reactance** from the magnetic vector potential (self and mutual inductance)
- **Capacitive effects** at high frequencies (self-resonance)

## PEEC formulation with Loop-Star decomposition

The PEEC method discretizes the surface current $\mathbf{J}$ on the conductor surface
into a set of basis functions. We use the **Loop-Star decomposition**, which separates
the current into:

- **Loop (solenoidal) basis** — divergence-free edge currents (Raviart-Thomas / RWG functions)
- **Star (irrotational) basis** — curl-free charge-producing currents (piecewise constant on cells)

### Block system

The Loop-Star PEEC system at angular frequency $\omega = 2\pi f$ reads:

$$
\begin{pmatrix}
R + j\omega L & M_{LS}^T \\
M_{LS} & P / (j\omega)
\end{pmatrix}
\begin{pmatrix}
I_\text{loop} \\
Q_\text{star}
\end{pmatrix}
=
\begin{pmatrix}
V_\text{port} \\
0
\end{pmatrix}
$$

where:
- $L_{ij} = \mu_0 \int\!\int \frac{\mathbf{J}_i(\mathbf{r}) \cdot \mathbf{J}_j(\mathbf{r}')}{4\pi|\mathbf{r}-\mathbf{r}'|} \, dS \, dS'$ — **inductance matrix** (via Laplace single-layer on `HDivSurface`)
- $P_{ij} = \frac{1}{\varepsilon_0} \int\!\int \frac{\phi_i(\mathbf{r}) \, \phi_j(\mathbf{r}')}{4\pi|\mathbf{r}-\mathbf{r}'|} \, dS \, dS'$ — **potential coefficient matrix** (via scalar single-layer on `SurfaceL2`)
- $M_{LS,ij} = \int \nabla\!\cdot\!\mathbf{J}_j \; \phi_i \, dS$ — **divergence coupling** (standard FEM bilinear form)
- $R = \text{diag}(R_\text{edge})$ — **resistance** from sheet resistance

### ngsbem implementation

The key advantage of using **ngsbem** is the natural mapping to NGSolve function spaces:

| PEEC quantity | NGSolve space | BEM operator |
|:---|:---|:---|
| Loop DOFs ($\mathbf{J}$) | `HDivSurface` (order $p$) | `LaplaceSL` |
| Star DOFs ($\phi$) | `SurfaceL2` (order $p$) | `SingleLayerPotentialOperator` |
| Coupling $M_{LS}$ | Product space | `div(u) * v * ds` (FEM) |

This provides Galerkin discretization (symmetric matrices by construction),
high-order elements, and FMM acceleration for large problems.

## 1. Setup and mesh generation

We create a rectangular plate conductor (10 mm × 10 mm, 35 µm copper)
representing a typical PCB trace or bus bar cross-section.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Add the Radia source path
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..', '..', 'src', 'radia'))

from ngbem_peec import NGBEMPEECSolver, create_plate_mesh, get_mesh_triangles

# Physical constants
MU_0 = 4.0 * np.pi * 1e-7   # H/m
EPS_0 = 8.854187817e-12      # F/m

In [ ]:
# Conductor geometry
width = 0.01       # 10 mm
height = 0.01      # 10 mm
maxh = 0.003       # ~3 mm element size
thickness = 35e-6  # 35 um (1 oz copper)
sigma = 5.8e7      # Copper conductivity [S/m]

# Generate surface mesh via Netgen OCC
mesh = create_plate_mesh(width, height, maxh, label="conductor")

# Extract triangles for visualization
triangles, areas = get_mesh_triangles(mesh)
n_tri = len(triangles)
total_area = np.sum(areas)

print(f"Conductor: {width*1e3:.0f} mm x {height*1e3:.0f} mm, t = {thickness*1e6:.0f} um")
print(f"Mesh: {n_tri} triangles, total area = {total_area*1e6:.2f} mm²")
print(f"Sheet resistance: R_sq = {1/(sigma*thickness)*1e3:.2f} mΩ/sq")

In [ ]:
# Visualize the mesh
from matplotlib.collections import PolyCollection

fig, ax = plt.subplots(1, 1, figsize=(6, 6))

verts_2d = [[[v[0]*1e3, v[1]*1e3] for v in tri] for tri in triangles]
pc = PolyCollection(verts_2d, edgecolors='k', facecolors='lightskyblue',
                     linewidths=0.8)
ax.add_collection(pc)
ax.set_xlim(-0.5, width*1e3 + 0.5)
ax.set_ylim(-0.5, height*1e3 + 0.5)
ax.set_aspect('equal')
ax.set_xlabel('x [mm]')
ax.set_ylabel('y [mm]')
ax.set_title(f'Conductor surface mesh ({n_tri} triangles)')
plt.tight_layout()
plt.show()

## 2. BEM matrix assembly

We assemble the PEEC matrices using ngsbem:

1. **Inductance $L$**: Laplace single-layer BEM operator on `HDivSurface` (edge DOFs)
2. **Potential coefficients $P$**: Scalar single-layer on `SurfaceL2` (cell DOFs)
3. **Loop-Star coupling $M_{LS}$**: Divergence bilinear form (FEM, not BEM)
4. **Resistance $R$**: From sheet resistance $R_\square = 1/(\sigma t)$

In [ ]:
# Create solver and assemble matrices
solver = NGBEMPEECSolver(mesh, conductor_label="conductor",
                          sigma=sigma, thickness=thickness,
                          order=0, intorder=5)
solver.assemble()

print(f"Loop DOFs (HDivSurface edges): {solver.n_loop}")
print(f"Star DOFs (SurfaceL2 cells):   {solver.n_star}")
print(f"Assembly time: {solver.t_assemble:.3f} s")

### Matrix properties

The Galerkin BEM matrices have important properties:
- $L$ is **symmetric positive semi-definite** (from the Laplace single-layer kernel)
- $P$ is **symmetric positive definite** (from the scalar single-layer potential)
- $M_{LS}$ is **sparse** (each edge connects at most 2 triangles)

In [ ]:
# Verify matrix properties
L_eig = np.linalg.eigvalsh(solver.L)
P_eig = np.linalg.eigvalsh(solver.P)
M_nnz = np.count_nonzero(np.abs(solver.M_LS) > 1e-15)

print("L (inductance):")
print(f"  Symmetry error ||L-L^T||/||L|| = {np.linalg.norm(solver.L - solver.L.T)/np.linalg.norm(solver.L):.2e}")
print(f"  Eigenvalue range: [{L_eig[0]:.4e}, {L_eig[-1]:.4e}] H")
print(f"  All eigenvalues >= 0: {np.all(L_eig >= -1e-15*np.max(np.abs(L_eig)))}")
print()
print("P (potential coefficients):")
print(f"  Symmetry error ||P-P^T||/||P|| = {np.linalg.norm(solver.P - solver.P.T)/np.linalg.norm(solver.P):.2e}")
print(f"  Eigenvalue range: [{P_eig[0]:.4e}, {P_eig[-1]:.4e}] 1/F")
print(f"  All eigenvalues > 0: {np.all(P_eig > 0)}")
print()
print(f"M_LS (coupling): {M_nnz} nonzeros / {solver.M_LS.size} entries")
print(f"  Sparsity: {(1 - M_nnz/solver.M_LS.size)*100:.1f}%")

## 3. MQS impedance sweep

In the **magneto-quasi-static (MQS)** regime, capacitive effects are negligible.
Only the Loop DOFs contribute, and the impedance reduces to:

$$Z_\text{MQS}(\omega) = \mathbf{e}^T (R + j\omega L)^{-1} \mathbf{e}$$

This is valid from DC up to the frequency where the wavelength becomes comparable
to the conductor size (typically up to ~1 MHz for mm-scale conductors).

In [ ]:
# MQS frequency sweep
freqs_mqs = np.logspace(3, 9, 50)  # 1 kHz to 1 GHz
Z_mqs = solver.solve_frequency(freqs_mqs, mode='mqs')

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Magnitude
axes[0].loglog(freqs_mqs, np.abs(Z_mqs), 'b-', linewidth=1.5)
axes[0].set_ylabel('|Z| [Ω]')
axes[0].set_title('MQS Impedance (Loop-only)')
axes[0].grid(True, which='both', alpha=0.3)

# Phase
axes[1].semilogx(freqs_mqs, np.angle(Z_mqs, deg=True), 'r-', linewidth=1.5)
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [deg]')
axes[1].set_ylim([-5, 95])
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Print a few values
for f, Z in zip(freqs_mqs[::10], Z_mqs[::10]):
    L_nH = np.imag(Z) / (2*np.pi*f) * 1e9
    print(f"  f = {f:.1e} Hz:  R = {Z.real:.4e} Ω,  L = {L_nH:.2f} nH,  |Z| = {np.abs(Z):.4e} Ω")

## 4. Full Loop-Star impedance (with capacitive effects)

Including the Star DOFs captures the capacitive self-resonance of the conductor.
The port impedance is obtained via the **Schur complement**:

$$
Z_\text{eff} = (R + j\omega L) - M_{LS}^T \left(\frac{P}{j\omega}\right)^{-1} M_{LS}
$$

At low frequencies, $P/(j\omega) \to \infty$ and the capacitive correction vanishes
(recovering MQS). At high frequencies, the capacitive term dominates and the
impedance transitions from inductive to capacitive — this is the **self-resonance**.

In [ ]:
# Full Loop-Star frequency sweep
freqs_full = np.logspace(3, 11, 80)  # 1 kHz to 100 GHz
Z_full = solver.solve_frequency(freqs_full, mode='full')

# Find self-resonance (where Im(Z) crosses zero from positive to negative)
X = np.imag(Z_full)
resonance_idx = None
for i in range(1, len(X)):
    if X[i-1] > 0 and X[i] <= 0:
        resonance_idx = i
        break

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

# Magnitude
axes[0].loglog(freqs_full, np.abs(Z_full), 'b-', linewidth=1.5, label='Full Loop-Star')
axes[0].loglog(freqs_mqs, np.abs(Z_mqs), 'g--', linewidth=1, alpha=0.7, label='MQS (Loop-only)')
if resonance_idx is not None:
    axes[0].axvline(freqs_full[resonance_idx], color='gray', linestyle=':', alpha=0.5)
axes[0].set_ylabel('|Z| [Ω]')
axes[0].set_title('Full Loop-Star vs MQS Impedance')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.3)

# Phase
axes[1].semilogx(freqs_full, np.angle(Z_full, deg=True), 'r-', linewidth=1.5)
if resonance_idx is not None:
    axes[1].axvline(freqs_full[resonance_idx], color='gray', linestyle=':',
                     alpha=0.5, label=f'SRF ≈ {freqs_full[resonance_idx]:.1e} Hz')
    axes[1].legend()
axes[1].set_xlabel('Frequency [Hz]')
axes[1].set_ylabel('Phase [deg]')
axes[1].set_ylim([-95, 95])
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

if resonance_idx is not None:
    f_res = freqs_full[resonance_idx]
    print(f"Self-resonance frequency: {f_res:.2e} Hz ({f_res/1e9:.1f} GHz)")
    print("(where impedance transitions from inductive to capacitive)")
else:
    print("No self-resonance found in the frequency range.")

## 5. High-order convergence

One advantage of the ngsbem approach is the ability to use **high-order elements**.
We compare results for polynomial order $p = 0, 1, 2$ on the same mesh.

In [ ]:
# Convergence study
f_test = 1e6  # 1 MHz
orders = [0, 1, 2]

print(f"Convergence study at f = {f_test/1e6:.0f} MHz:")
print(f"{'Order':>5s}  {'n_loop':>7s}  {'n_star':>7s}  {'t_asm [ms]':>10s}  {'L [nH]':>10s}  {'Change':>10s}")
print("-" * 60)

prev_L = None
for order in orders:
    s = NGBEMPEECSolver(mesh, conductor_label="conductor",
                         sigma=sigma, thickness=thickness,
                         order=order, intorder=5 + 2*order)
    s.assemble()
    Z = s.solve_frequency(np.array([f_test]), mode='mqs')
    L_nH = np.imag(Z[0]) / (2*np.pi*f_test) * 1e9

    change = ""
    if prev_L is not None:
        pct = abs(L_nH - prev_L) / abs(prev_L) * 100
        change = f"{pct:.1f}%"
    prev_L = L_nH

    print(f"{order:>5d}  {s.n_loop:>7d}  {s.n_star:>7d}  {s.t_assemble*1e3:>10.1f}  {L_nH:>10.3f}  {change:>10s}")

## Summary

This notebook demonstrated the PEEC impedance extraction workflow using ngsbem:

1. **Mesh generation**: Netgen OCC creates a surface mesh of the conductor
2. **BEM assembly**: ngsbem computes $L$ (inductance) and $P$ (potential coefficients)
   via Galerkin boundary elements on `HDivSurface` and `SurfaceL2`
3. **MQS sweep**: Loop-only impedance $Z = R + j\omega L$ for the power-electronics regime
4. **Full Loop-Star**: Includes capacitive coupling via Schur complement,
   revealing self-resonance
5. **High-order convergence**: Increasing FE order improves accuracy on the same mesh

### Key advantages of the ngsbem approach

- **Galerkin discretization** — symmetric matrices by construction
- **High-order elements** — convergence with $p$-refinement, not just $h$-refinement
- **Natural Loop-Star decomposition** — no explicit basis transformation needed;
  `HDivSurface` and `SurfaceL2` are the Loop and Star spaces
- **FMM acceleration** — available in ngsbem for large-scale problems

### References

- A. Ruehli, "Equivalent Circuit Models for Three-Dimensional Multiconductor Systems,"
  *IEEE Trans. MTT*, 1974.
- F. Andriulli et al., "A Multiplicative Calderon Preconditioner for the Electric Field
  Integral Equation," *IEEE Trans. AP*, 2008.
- J. Ostrowski et al., "ngbem — A BEM Library for NGSolve," 2024.